# Session 2: From Static Weights to Adaptive Allocation — Designing the AI Rebalancing Engine

In Session 1, we built a classical minimum-variance portfolio and watched it fail under stress. The core problem? Static weights can't respond to a changing world. In this session, we take humans _partially_ out of the loop: we design an AI rebalancing engine that reads market sentiment, adjusts risk preferences dynamically, and enforces explicit safety rules — all without manual intervention.

> **By the end of this session, you will be able to:**
> * Design a complete AI rebalancing engine with three phases: Initialization, Daily Loop, and Output
> * Construct a sentiment signal from EMA crossover that drives the engine's risk appetite
> * Implement a Single Index Model (SIM) allocator that translates sentiment into portfolio weights
> * Define and enforce trigger rules: drawdown limits, turnover caps, and de-risking protocols

Let's build a machine that adapts.

## Examples
The following example notebooks accompany this lecture. They contain executable code that implements the concepts discussed here:

➡ [Example: Building the Sentiment Signal and Rebalancing Engine](eCornell-AI-Finance-L2-Example-BuildRebalancingEngine-May-2026.ipynb). In this example, we generate a synthetic market path, compute the EMA crossover sentiment signal, build the SIM-based allocator, and run the rebalancing engine over a simulated trading year. We compare the adaptive engine's wealth curve to the static min-variance baseline from Session 1.

➡ [Example: Trigger Rules and Scorecard Comparison](eCornell-AI-Finance-L2-Example-TriggerRulesAndScorecard-May-2026.ipynb). In this example, we implement drawdown-based de-risking and turnover caps, test how these safety rules affect portfolio behavior during stress, and produce an updated scorecard that compares the rebalancing engine to Session 1's static baseline.

## Why Static Weights Fail

Session 1 showed us that the minimum-variance optimizer treats its inputs — the expected return vector $\boldsymbol{\mu}$ and the covariance matrix $\boldsymbol{\Sigma}$ — as fixed constants. But markets are not stationary. Correlations spike during crises, volatilities cluster, and expected returns shift with economic regimes. A portfolio computed from last year's data has no mechanism to react when the world changes _today_.

> **The core limitation.** Static portfolio construction is a _one-shot_ decision: estimate, optimize, deploy. There is no feedback loop. No matter what happens after deployment — a war, a rate hike, a pandemic — the weights don't move until a human intervenes. This is not a flaw in the math; it's a flaw in the _architecture_.

The fix is straightforward in concept: replace the one-shot decision with a _loop_ that continuously reads market conditions and adjusts the portfolio accordingly. The challenge is designing that loop so it is systematic, transparent, and safe. That's what the rebalancing engine is for.

## The Sentiment Signal: EMA Crossover → λ

Before the engine can decide _what_ to do, it needs to know _what's happening_. We construct a real-time sentiment signal from the market price itself using exponential moving average (EMA) crossover — the same technique used in the CHEME-5660 TradeBot.

> **Exponential Moving Average.** The EMA of a price series $S_t$ with window $N$ is:
>
> $$\bar{S}_{t} = \alpha\, S_{t} + (1 - \alpha)\,\bar{S}_{t-1}, \qquad \alpha = \frac{2}{N + 1}$$
>
> We compute two EMAs: a **short-term** ($N_0 = 21$ days, ~1 month) that tracks recent momentum, and a **long-term** ($N_1 = 63$ days, ~1 quarter) that tracks the underlying trend.

The crossover between short and long EMAs produces a sentiment signal $\lambda_t$ that drives the engine's risk appetite:

> **The Lambda Signal.**
>
> $$\lambda_t = -G \cdot \left(\frac{\bar{S}^{\text{short}}_t}{\bar{S}^{\text{long}}_t} - 1\right)$$
>
> where $G > 0$ is a gain constant that controls sensitivity. The interpretation is simple:
> * $\lambda_t > 0$ → **Bearish** (short EMA below long EMA — recent prices falling relative to trend). The engine becomes risk-averse.
> * $\lambda_t < 0$ → **Bullish** (short EMA above long EMA — recent prices rising). The engine takes on more risk.
> * $\lambda_t \approx 0$ → **Neutral** (EMAs converging — no strong signal).

This signal is computed from a broad market index (e.g., SPY) and applied to the entire portfolio. It is a _market-level_ sentiment indicator, not a stock-specific one. In Session 4, we'll explore adding stock-level sentiment from news and filings.

## The SIM Allocator: From Sentiment to Shares

The lambda signal tells us _how much_ risk to take. The Single Index Model (SIM) allocator translates that risk appetite into _which assets to hold and how many shares_.

Each asset $i$ has SIM parameters $(\alpha_i, \beta_i, \sigma_i)$ estimated from historical data, where:
* $\alpha_i$ is the asset's idiosyncratic return (alpha)
* $\beta_i$ is the asset's sensitivity to the market factor
* $\sigma_i$ is the residual volatility

> **The SIM Allocation Rule.** Given the current sentiment $\lambda_t$ and EMA-smoothed market growth $g_{m,t}$, the allocator computes a risk-adjusted expected growth for each asset:
>
> $$\hat{g}_i = \frac{\alpha_i}{\beta_i^{\lambda_t}} + \frac{\beta_i}{\beta_i^{\lambda_t}} \cdot g_{m,t}$$
>
> The risk factor $\beta_i^{\lambda_t}$ is the key mechanism: when $\lambda_t > 0$ (bearish), high-beta assets are penalized exponentially; when $\lambda_t < 0$ (bullish), they are amplified. The preference weight is then:
>
> $$\gamma_i = \tanh(\hat{g}_i)$$
>
> which maps the growth score to $(-1, 1)$.

**Allocation logic:**
* **Preferred assets** ($\gamma_i > 0$): Receive budget proportionally: $n_i = \frac{\gamma_i}{\bar{\gamma}} \cdot \frac{B}{P_i}$, where $\bar{\gamma} = \sum_{j:\gamma_j > 0} \gamma_j$.
* **Non-preferred assets** ($\gamma_i \leq 0$): Pinned to a minimum position $\epsilon$ (keeps them in the portfolio for potential recovery).

This is the same allocation mechanism used by the CHEME-5660 TradeBot. The critical insight is that $\lambda_t$ acts as a _dial_ — it continuously adjusts the portfolio's risk exposure based on current market conditions, without requiring a human to re-estimate $\boldsymbol{\mu}$ and $\boldsymbol{\Sigma}$.

## The Rebalancing Engine: Pseudo-code

Putting it all together, the AI rebalancing engine operates in three phases:

> **Phase 1: Initialization**
> 1. Generate (or load) synthetic market price path $S_t$ for a broad index (e.g., SPY).
> 2. Compute short-term EMA ($N_0 = 21$) and long-term EMA ($N_1 = 63$). Define warmup offset $t_0 = N_0 + N_1$.
> 3. From EMA crossover, build the sentiment series: $\lambda_t = -G \cdot (\bar{S}^{\text{short}}_t / \bar{S}^{\text{long}}_t - 1)$.
> 4. Smooth market growth into $g_{m,t}$ using an EMA of log returns ($N_2 = 10$).
> 5. Assemble the **context model**: initial budget $B$, ticker universe, price matrix, SIM parameters $(\alpha_i, \beta_i, \sigma_i)$, risk floor $\epsilon$, time step $\Delta t$, and market factor $g_{m,t}$.
> 6. Define **trigger rules**: drawdown limit, turnover cap, and reallocation schedule $a_t \in \{0, 1\}$.
> 7. Compute initial allocation after warmup period.

> **Phase 2: Daily Loop** (for $t = t_0 + 1, \ldots, t_0 + T$)
> 1. Read reallocation flag $a_t$ from schedule.
> 2. **If $a_t = 1$ (rebalance day):**
>    - Liquidate current positions at today's prices: $B_t = \text{cash}_{t-1} + \sum_i n_{i,t-1} \cdot P_{i,t}$
>    - **Check drawdown trigger:** if $(W^{\text{peak}} - B_t) / W^{\text{peak}} > d_{\max}$, de-risk to 100% cash
>    - Otherwise, set $\lambda = \lambda_{t-1}$, run SIM allocator for new positions
>    - **Check turnover cap:** if trade volume $> \tau_{\max}$, scale trades proportionally
> 3. **If $a_t = 0$ (hold day):** Propagate prior allocation unchanged.

> **Phase 3: Output**
> Return history indexed by trading day: positions $n_t$, prices $P_t$, preference weights $\gamma_t$, unallocated cash, and total wealth $W_t$.

This is a _deterministic_ engine — given the same inputs and rules, it produces the same output every time. The "AI" is in the _design_ of the signal ($\lambda$), the allocation rule (SIM), and the safety triggers — not in opaque black-box decisions.

## Trigger Rules: The Safety Net

An adaptive engine that trades freely is dangerous — it can churn the portfolio, chase noise, or fail to protect capital during a crash. Trigger rules are the _guardrails_ that keep the engine safe:

| Rule | Parameter | What It Does |
|------|-----------|-------------|
| **Drawdown Limit** | $d_{\max}$ (e.g., 10%) | If portfolio wealth drops more than $d_{\max}$ from its peak, the engine de-risks to 100% cash. This is a circuit breaker — it stops the bleeding. |
| **Turnover Cap** | $\tau_{\max}$ (e.g., 50%) | If the proposed rebalance would trade more than $\tau_{\max}$ of portfolio value, the trades are scaled down proportionally. This controls transaction costs. |
| **Reallocation Schedule** | $a_t \in \{0, 1\}$ | The binary schedule determines _which days_ the engine is allowed to rebalance. Daily ($a_t = 1$ always), weekly, or event-driven. |

> **Why are trigger rules essential?** Without them, the engine is unconstrained — it could rebalance every day (expensive), ignore a crash (catastrophic), or flip positions wildly based on noise in the lambda signal. Trigger rules encode _human judgment_ about acceptable risk into the machine's decision loop. They are the bridge between autonomous operation and investment committee oversight.

In Session 4, we'll extend these rules with human override protocols and escalation procedures for production deployment.

## Summary

> **Key Takeaways from Session 2:**
> * Static portfolio weights are a one-shot decision with no feedback loop — the rebalancing engine replaces this with a continuous adaptive loop
> * The EMA crossover signal $\lambda_t$ provides a simple, interpretable measure of market sentiment that drives the engine's risk appetite
> * The SIM allocator uses $\lambda_t$ to dynamically adjust asset preferences: penalizing high-beta assets when bearish, amplifying them when bullish
> * Trigger rules (drawdown limits, turnover caps, reallocation schedules) encode human safety judgment into the machine, preventing catastrophic outcomes

**Next Session:** In [Session 3](../session-3/L3/eCornell-AI-Finance-L3-Lecture-HMMBacktesting-May-2026.ipynb), we'll stress-test this rebalancing engine against Hidden Markov Model-generated synthetic market paths that include regime shifts — normal markets, stressed markets, and crashes. Can the engine survive what the static portfolio couldn't?

### Disclaimer
This content is for educational purposes only and does not constitute investment advice. The rebalancing engine described here is a pedagogical tool using synthetic data and simplified models. Real-world algorithmic trading involves regulatory requirements, execution risk, and operational considerations beyond the scope of this session.